# The Golden Age of Video Games

The video game industry moves a lot of money: according to Mordor Intelligence, the global market is projected to exceed $300 billion by 2027. With that much money at stake, major publishers have every incentive to produce the next big hit.
That made me wonder: are video games actually getting better over time, or has the so-called "golden age" of gaming already come and gone?
To answer this question, I decided to analyze critic and user review scores, along with sales data, for the top 400 video games released since 1977. My plan was to combine (JOIN) different tables, compare results, and filter, group, and order the data to find patterns across the years.
The database I worked with contains the following tables (each limited to 400 rows for this project; the full dataset, with over 13,000 games, is available on Kaggle):


### `game_sales` table

| Column | Definition | Data Type |
|-|-|-|  
|name|Name of the video game|`varchar`|
|platform|Gaming platform|`varchar`|
|publisher|Game publisher|`varchar`|
|developer|Game developer|`varchar`|
|games_sold|Number of copies sold (millions)|`float`|
|year|Release year|`int`|

### `reviews` table

| Column | Definition | Data Type |
|-|-|-|
|name|Name of the video game|`varchar`|  
|critic_score|Critic score according to Metacritic|`float`|
|user_score|User score according to Metacritic|`float`|


### `users_avg_year_rating` table

| Column | Definition | Data Type |
|-|-|-|
|year| Release year of the games reviewed |`int`|  
|num_games| Number of games released that year |`int`|
|avg_user_score| Average score of all the games ratings for the year |`float`|

### `critics_avg_year_rating` table

| Column | Definition | Data Type |
|-|-|-|
|year| Release year of the games reviewed |`int`|  
|num_games| Number of games released that year |`int`|
|avg_critic_score| Average score of all the games ratings for the year |`float`|



## Step 1: What are the best-selling games of all time?
Before looking at the ratings, I wanted to start with the business side. Which games have sold the most copies, regardless of how well or poorly critics rated them? This gives me an initial "sales top" context against which I'll later compare perceived quality.


In [ ]:
-- best_selling_games
SELECT *
FROM game_sales
ORDER BY games_sold DESC
LIMIT 10;


,name,platform,publisher,developer,games_sold,year
0,Wii Sports for Wii,Wii,Nintendo,Nintendo EAD,82.90,2006
1,Super Mario Bros. for NES,NES,Nintendo,Nintendo EAD,40.24,1985
2,Counter-Strike: Global Offensive for PC,PC,Valve,Valve Corporation,40.00,2012
3,Mario Kart Wii for Wii,Wii,Nintendo,Nintendo EAD,37.32,2008
4,PLAYERUNKNOWN'S BATTLEGROUNDS for PC,PC,PUBG Corporation,PUBG Corporation,36.60,2017
5,Minecraft for PC,PC,Mojang,Mojang AB,33.15,2010
6,Wii Sports Resort for Wii,Wii,Nintendo,Nintendo EAD,33.13,2009
7,Pokemon Red / Green / Blue Version for GB,GB,Nintendo,Game Freak,31.38,1998
8,New Super Mario Bros. for DS,DS,Nintendo,Nintendo EAD,30.80,2006
9,New Super Mario Bros. Wii for Wii,Wii,Nintendo,Nintendo EAD,30.30,2009


## Step 2: Which years had the best critic scores?
With the commercial picture in place, I moved on to the central question of the project: identifying the years in which critics rated the games released that year the highest. To do this, I calculated the average critic_score per year by joining game_sales and reviews, and filtered the results to only include years with at least 4 reviewed games (to avoid a single highly-rated game skewing an entire year's average).


In [ ]:
-- critics_top_ten_years
SELECT u.year AS year, u.num_games AS num_games, sub.cs AS avg_critic_score
FROM users_avg_year_rating AS u,
    (SELECT g.year, ROUND(AVG(r.critic_score), 2) AS cs
     FROM game_sales AS g
     INNER JOIN reviews AS r
     USING (name)
     GROUP BY g.year) AS sub
WHERE u.year=sub.year AND u.num_games >=4
ORDER BY avg_critic_score DESC
LIMIT 10;

,year,num_games,avg_critic_score
0,1998,10,9.32
1,2004,11,9.03
2,2002,9,8.99
3,1999,11,8.93
4,2001,13,8.82
5,2011,26,8.76
6,2016,13,8.67
7,2013,18,8.66
8,2008,20,8.63
9,2012,12,8.62


## Step 3: Searching for the true "golden age"
I already had the top years according to critics. However, users and critics don't always agree. So, I wanted to find the years that were truly exceptional — that is, years where at least one of the two groups (critics or users) gave the games released that year an average score above 9. To do this, I combined both yearly average tables with a FULL JOIN (so I wouldn't lose any year that stood out in just one of the two metrics), and I also calculated the difference between both scores to see how aligned critics and users actually were during those golden years.


In [ ]:
-- golden_years
SELECT year, num_games, avg_critic_score, avg_user_score, (avg_critic_score-avg_user_score) AS diff
FROM critics_avg_year_rating
FULL JOIN users_avg_year_rating
USING (year, num_games)
WHERE avg_critic_score > 9 OR avg_user_score >9
ORDER BY year 

,year,num_games,avg_critic_score,avg_user_score,diff
0,1997,8,7.93,9.50,-1.57
1,1998,10,9.32,9.40,-0.08
2,2004,11,9.03,8.55,0.48
3,2008,20,8.63,9.03,-0.40
4,2009,20,8.55,9.18,-0.63
5,2010,23,8.41,9.24,-0.83


## Conclusions
After going through these three queries, I can highlight the following points from the analysis:

•	Commercial success and critical success don't always go hand in hand. The best-selling games of all time (Step 1) don't necessarily overlap with the years critics rated the highest (Step 2), confirming that sales and perceived quality are two separate stories within the industry.

•	A "golden age" does appear to exist. By cross-referencing critic and user averages (Step 3), a small group of years emerges where both audiences, or at least one of them, rated the games released that year exceptionally well, above 9. These years are the strongest candidates for the industry's peak quality period. For example, when grouping the data by decade, the years between 2000 and 2010 show a high volume of sales along with fairly high scores from both critics and users.

•	The diff column is key to understanding consensus. When the difference between avg_critic_score and avg_user_score is small, it means critics and the public agreed that year was exceptional; when the difference is large, it suggests the "golden age" depends on who you ask.

Overall, the analysis suggests that the golden age of video games isn't a myth or a single fixed era: it's a handful of specific years where, according to the data, the industry reached quality peaks that didn't always coincide with its biggest sales peaks.
